# LLM Multi-Turn — Conversation Memory + Token Tracking

Maintain a running **message history** so the model can refer back to earlier turns. After each API call, print **latency** and **token usage** — the same metrics as `LiteLLM_Test_02`, but using native SDKs.

The conversation covers three turns; the model's final reply should summarize what was discussed.

In [ ]:
import time
import os

TURNS = [
    "I'm learning about neural networks. What is a perceptron?",
    'How does that connect to backpropagation?',
    'Summarize everything we just covered in two sentences.'
]

def print_metrics(latency: float, prompt_tok: int, completion_tok: int):
    total = prompt_tok + completion_tok
    print(f'  [latency] {latency:.2f}s  '
          f'[tokens] prompt={prompt_tok}, completion={completion_tok}, total={total}')

---
## Ollama (local)

Same append pattern as OpenAI. Token counts come from `resp['prompt_eval_count']` and `resp['eval_count']`.

In [ ]:
import ollama

OLLAMA_MODEL = 'mistral-nemo:12b-instruct-2407-q4_K_M'

messages = []

for turn in TURNS:
    messages.append({'role': 'user', 'content': turn})
    print(f'User: {turn}')

    t0 = time.time()
    resp = ollama.chat(model=OLLAMA_MODEL, messages=messages)
    dt = time.time() - t0

    reply = resp['message']['content']
    messages.append({'role': 'assistant', 'content': reply})
    print(f'Assistant: {reply}')
    print_metrics(dt, resp.get('prompt_eval_count', 0), resp.get('eval_count', 0))
    print()

---
## OpenAI

Build a `messages` list; append each assistant reply before the next user turn. Token counts come from `resp.usage`.

In [ ]:
from openai import OpenAI

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
OPENAI_MODEL   = 'gpt-5'

openai_client = OpenAI(api_key=OPENAI_API_KEY)
messages = []

for turn in TURNS:
    messages.append({'role': 'user', 'content': turn})
    print(f'User: {turn}')

    t0 = time.time()
    resp = openai_client.chat.completions.create(
        model=OPENAI_MODEL, messages=messages
    )
    dt = time.time() - t0

    reply = resp.choices[0].message.content
    messages.append({'role': 'assistant', 'content': reply})
    print(f'Assistant: {reply}')
    print_metrics(dt, resp.usage.prompt_tokens, resp.usage.completion_tokens)
    print()

---
## Anthropic

Same pattern. Anthropic's `usage` attribute exposes `input_tokens` and `output_tokens`.

In [ ]:
from anthropic import Anthropic

ANTHROPIC_API_KEY = os.getenv('ANTHROPIC_API_KEY')
ANTHROPIC_MODEL   = 'claude-sonnet-4-5-20250929'

anthropic_client = Anthropic(api_key=ANTHROPIC_API_KEY)
messages = []

for turn in TURNS:
    messages.append({'role': 'user', 'content': turn})
    print(f'User: {turn}')

    t0 = time.time()
    resp = anthropic_client.messages.create(
        model=ANTHROPIC_MODEL, max_tokens=500, messages=messages
    )
    dt = time.time() - t0

    reply = resp.content[0].text
    messages.append({'role': 'assistant', 'content': reply})
    print(f'Assistant: {reply}')
    print_metrics(dt, resp.usage.input_tokens, resp.usage.output_tokens)
    print()

---
## Google Gemini

Use `model.start_chat()` to maintain history automatically. Token counts come from `resp.usage_metadata`.

In [ ]:
import google.generativeai as genai

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
GOOGLE_MODEL   = 'gemini-2.5-flash'

genai.configure(api_key=GEMINI_API_KEY)
gemini_model = genai.GenerativeModel(GOOGLE_MODEL)
chat = gemini_model.start_chat()

for turn in TURNS:
    print(f'User: {turn}')

    t0 = time.time()
    resp = chat.send_message(turn)
    dt = time.time() - t0

    print(f'Assistant: {resp.text}')
    u = resp.usage_metadata
    print_metrics(dt, u.prompt_token_count, u.candidates_token_count)
    print()